In [1]:
from langchain_openai import ChatOpenAI, OpenAI
from langchain.prompts.chat import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.prompts import PromptTemplate

# SystemMessagePromptTemplate - gives instructions, content or data for the AI model
# HumanMessagePromptTemplate - messages from user that AI responds to.

llm_chat = ChatOpenAI(model_name='gpt-4', temperature=0)

In [8]:
from langchain.chains import create_extraction_chain

schema = {
    "properties": {
        "news_article_title": {"type": "string"},
        "news_article_summary": {"type": "string"},
    },
    "required": ["news_article_title", "news_article_summary"],
}


def extract(content: str, schema: dict):
    return create_extraction_chain(schema=schema, llm=llm_chat).run(content)

In [9]:
async def main():
    print(1)
    
await main()

1


In [14]:
import pprint

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import AsyncChromiumLoader
from langchain_community.document_transformers import BeautifulSoupTransformer


async def scrape_with_playwright(urls, schema):
    loader = AsyncChromiumLoader(urls)
    docs = loader.load()
    bs_transformer = BeautifulSoupTransformer()
    docs_transformed = bs_transformer.transform_documents(
        docs, tags_to_extract=["span"]
    )
    print("Extracting content with LLM")

    # Grab the first 1000 tokens of the site
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=1000, chunk_overlap=0
    )
    splits = splitter.split_documents(docs_transformed)

    # Process the first split
    extracted_content = extract(schema=schema, content=splits[0].page_content)
    pprint.pprint(extracted_content)
    return extracted_content




urls = ["https://finance.yahoo.com/quote/TGT/news/"]
extracted_content = scrape_with_playwright(urls, schema=schema)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [4]:
template = 'you are an assistant that helps users find information about the movie'
systemMessagePrompt = SystemMessagePromptTemplate.from_template(template)
human_template = 'Find information about the movie {movie_title}'
humanMessagePrompt = HumanMessagePromptTemplate.from_template(human_template)

chat_promt = ChatPromptTemplate.from_messages([systemMessagePrompt, humanMessagePrompt])

'''
to_messages object in LangChain allows you to convert the formatted value 
of a chat prompt template into a list of message objects
''' 

llm_chat(chat_promt.format_prompt(movie_title='inception').to_messages())

/usr/local/lib/python3.9/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The function `__call__` was deprecated in LangChain 0.1.7 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(


AIMessage(content='"Inception" is a 2010 science fiction action film directed and written by Christopher Nolan. The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating the subconscious of his targets. He is offered a chance to have his criminal history erased as payment for the implantation of another person\'s idea into a target\'s subconscious. The ensemble cast includes Ken Watanabe, Joseph Gordon-Levitt, Marion Cotillard, Ellen Page, Tom Hardy, Dileep Rao, Cillian Murphy, Tom Berenger, and Michael Caine.\n\nThe film explores the themes of dreams, reality, and the human subconscious. It is known for its complex narrative structure, which involves multiple layers of dreams within dreams. The film was a commercial success, grossing over $800 million worldwide, and received critical acclaim for its screenplay, visual effects, score, and the performances of its cast. It won four Academy Awards for Best Cinematography, Best Sound Editing, Best Soun

## summarizer

In [12]:
from langchain.document_loaders import PyPDFLoader
from langchain.chains.summarize import load_summarize_chain

llm = OpenAI(model='gpt-3.5-turbo-instruct', temperature=0)

summarize_chain=load_summarize_chain(llm)
doc_loader = PyPDFLoader(file_path='/Users/rc/workspaces/llm/data/Policy.pdf')

doc = doc_loader.load()

summary = summarize_chain(doc)
print(summary['output_text'])

 This document is an auto policy endorsement for policy number 975916277, belonging to Raghunadh Chilukamari. It includes amendments to the policy, such as adding an exclusion for drivers not listed on the declarations page and adding trip interruption coverage. It also includes changes to the definitions, exclusions, and general provisions of the policy. The document also states that the policy may be cancelled or nonrenewed for various reasons, and that coverage may be denied for fraudulent or misrepresented information on the application. 


In [14]:
import os, requests, newspaper
from newspaper import Article
from langchain import PromptTemplate, LLMChain, FewShotPromptTemplate


headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.82 Safari/537.36'
}

article_url = 'https://www.artificialintelligence-news.com/2024/03/04/ai-india-need-government-permission-before-launching/'
session = requests.session()

try:
    response = session.get(article_url, headers=headers,timeout=10)
    if response.status_code==200:
        article = Article(article_url)
        article.download()
        article.parse()
        
#         print(f"Title: {article.title}")
#         print(f"Text: {article.text}")
    else:
        print(f"Failed to fetch article at {article_url}")
except Exception as e:
    print(f"Error occurred while fetching article at {article_url}: {e}")
    
    
article_title = article.title
article_text = article.text

template = """
You are a very good assistant that summarizes online articles.

Here's the article you want to summarize.

==================
Title: {article_title}

{article_text}
==================

Write a summary of the previous article.
"""

systemMessagePrompt = SystemMessagePromptTemplate.from_template(template)

human_template = 'summarize the following article with {article_title} and {article_text}'
humanMessagePrompt = HumanMessagePromptTemplate.from_template(human_template)

chat_promt = ChatPromptTemplate.from_messages([systemMessagePrompt, humanMessagePrompt])

llm_chat(chat_promt.format_prompt(article_title=article_title,article_text=article_text).to_messages())


AIMessage(content='India\'s Ministry of Electronics and Information Technology (MeitY) has issued an advisory stating that any AI technology still in development must obtain explicit government permission before being released to the public. The advisory also mandates that developers label the potential fallibility of the AI\'s output and implement a "consent popup" mechanism to inform users about potential defects or errors. It also requires the labelling of deepfakes to prevent misuse. The advisory, which is not legally binding, also orders all platforms to ensure that AI models do not permit bias, discrimination, or threaten the electoral process. Developers are asked to comply with the advisory within 15 days of its issuance. IT minister Rajeev Chandrasekhar stated that this stance would eventually be encoded in legislation.')

In [17]:
template = """
As an advanced AI, you've been tasked to summarize online articles into bulleted points. Here are a few examples of how you've done this in the past:

Example 1:
Original Article: 'The Effects of Climate Change
Summary:
- Climate change is causing a rise in global temperatures.
- This leads to melting ice caps and rising sea levels.
- Resulting in more frequent and severe weather conditions.

Example 2:
Original Article: 'The Evolution of Artificial Intelligence
Summary:
- Artificial Intelligence (AI) has developed significantly over the past decade.
- AI is now used in multiple fields such as healthcare, finance, and transportation.
- The future of AI is promising but requires careful regulation.

Now, here's the article you need to summarize:

==================
Title: {article_title}

{article_text}
==================

Please provide a summarized version of the article in a bulleted list format.
"""

prompt = template.format(article_text=article_text, article_title=article_title)

messages = [
    HumanMessage(content=prompt)
]


from langchain_openai import ChatOpenAI

llm_chat = ChatOpenAI(model='gpt-4', temperature = 0)

summary = llm_chat(messages)
print(summary.content)

- India's Ministry of Electronics and Information Technology (MeitY) has issued an advisory requiring AI technologies in development to obtain government permission before public release.
- Developers must label the potential fallibility or unreliability of the AI's output before deployment.
- The advisory also plans for a "consent popup" to inform users about potential AI errors and mandates the labelling of deepfakes to prevent misuse.
- All platforms are ordered to ensure that AI models, including large language models (LLM), do not permit bias, discrimination, or threaten electoral integrity.
- Developers are asked to comply with the advisory within 15 days and may need to perform a demo or undergo stress testing for government officials after application for product release.
- While the advisory is not legally binding, it indicates the government's expectations and the potential future direction of AI regulation.
- IT minister Rajeev Chandrasekhar stated that the advisory's stance

In [24]:
from langchain.output_parsers import PydanticOutputParser
from pydantic import validator
from pydantic import BaseModel, Field
from typing import List

class ArticleSummary(BaseModel):
    title: str = Field(description="Title of the article")
    summary: List[str] = Field(description="Bulleted list summary of the article")

    # validating whether the generated summary has at least three lines
    @validator('summary', allow_reuse=True)
    def has_three_or_more_lines(cls, list_of_lines):
        if len(list_of_lines) < 3:
            raise ValueError("Generated summary has less than three bullet points!")
        
        return list_of_lines

    
parser = PydanticOutputParser(pydantic_object=ArticleSummary)

template = """
You are a very good assistant that summarizes online articles.

Here's the article you want to summarize.

==================
Title: {article_title}

{article_text}
==================

{format_instructions}
"""


prompt = PromptTemplate(
    template = template,
    input_variables = ["article_title", "article_text"],
    partial_variables = {'format_instructions' : parser.get_format_instructions()}
)

formatted_prompt = prompt.format_prompt(article_title=article_title, article_text=article_text)
llm = OpenAI(model='gpt-3.5-turbo-instruct', temperature=0)

response = llm(formatted_prompt.to_string())
print(response)



/var/folders/2w/0kqpwndd7394lk_g4l7487jc0000gn/T/ipykernel_25563/1557022875.py:11: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.5/migration/
  @validator('summary', allow_reuse=True)


ValidationError: 1 validation error for PydanticOutputParser
pydantic_object
  subclass of BaseModel expected (type=type_error.subclass; expected_class=BaseModel)

In [16]:
from langchain.schema import (
                            SystemMessage,
                            HumanMessage,
                            AIMessage)

messages = [
    SystemMessage(content='You are a helpful assistant'),
    HumanMessage(content='what is the capital city of france?'),
    AIMessage(content='The capital city of france is paris')
]


prompt = HumanMessage(content="I wouldd like to know more about the capital city you just mentioned.")

messages.append(prompt)


response = llm_chat(messages)
print(response.content)

Paris, the capital city of France, is one of the most important and influential cities in the world. It is located in the north-central part of the country.

Paris is known for its stunning architecture, vibrant culture, and rich history. It is home to many world-renowned landmarks such as the Eiffel Tower, Notre-Dame Cathedral, Louvre Museum, and the Champs-Élysées. The city is also famous for its cuisine, fashion, and luxury goods.

Paris is often referred to as "The City of Light" because it was one of the first cities in the world to have street lighting. It's also known as "The City of Love" due to its romantic appeal.

The city is a major hub for art, science, and education. It hosts several world-class educational institutions and research centers. Paris is also a key player in the global economic landscape, with many international corporations having their headquarters there.

The Seine River runs through the city, dividing it into two parts: the Right Bank to the north and the

In [25]:
llm

OpenAI(client=<openai.resources.completions.Completions object at 0x111f98700>, async_client=<openai.resources.completions.AsyncCompletions object at 0x111f6d520>, temperature=0.0, openai_api_key=SecretStr('**********'), openai_proxy='')